# **The path, not only the point: Blur IG and Guided IG**

A practice for the module ["Attribution from axioms"](https://open-xai-platform.web.app).

The lesson says something important: a path integral has **two knobs** — where we start from
(the baseline) and how we travel (the path). Expected Gradients turn the first, Guided IG only
the second, Blur IG changes both.

Hence a consequence that is easy to state and impossible to remember without a number:
**Guided IG does not fix the blind spot of a black baseline** — it does not touch the baseline
at all. We are about to see this in percentages.

It runs on a CPU in a couple of minutes: three methods with 32 steps each.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'
STEPS = 32                                                  # integration steps — enough for completeness to converge within a percent
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)

tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
raw = urllib.request.urlopen(f'{DATA}/data/cat.jpg', timeout=30).read()
img = tf(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)     # the picture is in [0,1]: the normalization happens inside logit


def logit(x, c):
    """The class logit. Normalization inside, so that the path is built in picture space."""
    return model((x - MEAN) / STD)[0, c]


def grad(x, c):
    x = x.clone().requires_grad_(True)
    logit(x, c).backward()
    return x.grad


CLS = int(model((img - MEAN) / STD).argmax())
print(f'class: {CLS}')

## 1. What the Blur IG path looks like

Plain IG travels in a straight line in pixel space: from a black image to ours, linearly. Blur
IG travels differently — from a heavily blurred picture to the sharp one, gradually lowering the
blur radius. Let us look at that path.

In [ ]:
def blur(x, sigma):
    """Gaussian blur of radius sigma: two one-dimensional passes instead of one two-dimensional."""
    if sigma < 0.1:
        return x.clone()
    k = int(4 * sigma) | 1
    t = torch.arange(k, dtype=torch.float32) - k // 2
    g = torch.exp(-t ** 2 / (2 * sigma ** 2))
    g = (g / g.sum()).view(1, 1, -1)
    y = F.conv2d(F.pad(x, (k // 2,) * 2 + (0, 0), mode='reflect'),
                 g.expand(3, 1, 1, k), groups=3)
    return F.conv2d(F.pad(y, (0, 0) + (k // 2,) * 2, mode='reflect'),
                    g.view(1, 1, -1, 1).expand(3, 1, k, 1), groups=3)


fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, s in zip(axes, (20.0, 10.0, 5.0, 2.0, 0.0)):
    ax.imshow(blur(img, s)[0].permute(1, 2, 0).numpy())
    ax.set_title(f'sigma = {s:.0f}')
    ax.axis('off')
plt.tight_layout()
plt.show()

Note this: **all along the path the picture stays a picture.** That is the argument
for such a path. The middle of a straight path is a half-transparent grey smear that does not
occur in nature, and gradients there are noisy; on a blur path every intermediate point looks
like a real image, merely shot out of focus.

In [ ]:
def ig(x, baseline, c, steps=STEPS):
    """Plain IG: a straight path from the baseline to the object."""
    total = torch.zeros_like(x)
    for k in range(1, steps + 1):
        total += grad(baseline + (k / steps) * (x - baseline), c)
    return (x - baseline) * total / steps


def blur_ig(x, c, sigma_max=20.0, steps=STEPS):
    """Blur IG: the path runs from a heavily blurred picture to the sharp one, by lowering sigma."""
    sigmas = torch.linspace(sigma_max, 0.0, steps + 1)
    total, prev = torch.zeros_like(x), blur(x, float(sigmas[0]))
    for s in sigmas[1:]:
        cur = blur(x, float(s))
        total += grad(0.5 * (prev + cur), c) * (cur - prev)      # the gradient at the middle of the segment, times the shift along the path
        prev = cur
    return total


def guided_ig(x, baseline, c, steps=STEPS, q=0.5):
    """Guided IG: at each step only the features with the smallest gradient magnitude move."""
    cur, total = baseline.clone(), torch.zeros_like(x)
    for k in range(steps):
        g = grad(cur, c)
        remaining = baseline + ((k + 1) / steps) * (x - baseline) - cur
        mask = (g.abs() <= torch.quantile(g.abs().flatten(), q)).float()   # the bottom q percent by gradient magnitude — the function changes more calmly there
        step = remaining * mask + remaining * (1 - mask) * (k + 1 == steps)
        total += g * step
        cur = cur + step
    return total


zero = torch.zeros_like(img)
a_ig, a_blur, a_guided = ig(img, zero, CLS), blur_ig(img, CLS), guided_ig(img, zero, CLS)
print(f'computed')

## 2. Completeness: did it converge

All three methods are path integrals, and for all three the sum of the attributions must equal
the difference of the logits between the ends of the path. Let us check.

In [ ]:
f_x = logit(img, CLS).item()
for name, a, ref in (('IG', a_ig, zero),
                     ('Blur IG', a_blur, blur(img, 20.0)),
                     ('Guided IG', a_guided, zero)):
    print(f'   {name:11} sum of attributions {a.sum():+7.3f}   f(x) - f(reference) '
          f'{f_x - logit(ref, CLS).item():+7.3f}')

It converges for all three, to within a percent — that is the error of the Riemann
sum at 32 steps, and it shrinks as their number grows.

**Look at the reference point in the third column.** For IG and Guided IG it is zero, for Blur
IG it is the blurred picture. The numbers therefore differ, and that is not a discrepancy but
**different questions**: "why not emptiness" versus "what does the detail add over the general
composition". Exactly what the lesson says in the section on baselines.

**Task 1.** Compute completeness at 8, 32 and 128 steps and see how the residual shrinks. Does
it shrink monotonically?

In [ ]:
# Your code here

## 3. The main thing: only one of the two fixes the blind spot

The lesson explains that a black baseline has a blind spot: a pixel that coincides with the
baseline gets a strictly zero attribution, because the factor $(x_i - x'_i)$ zeroes everything
out. So the dark regions of an image drop out of the explanation.

Let us measure it. Take the dark pixels and see what share of the attribution magnitude falls
on them.

In [ ]:
dark = (img.mean(1, keepdim=True) < 0.25).float()
print(f'dark regions occupy this share of the picture area: {dark.mean().item() * 100:.1f} %\n')
for name, a in (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided)):
    mass = a.abs().sum(1, keepdim=True)
    print(f'   {name:11} {(mass * dark).sum().item() / mass.sum().item() * 100:5.1f} %')

This table is what the whole thing was for.

- **IG with a zero baseline** gave the dark regions about a percent and a half of the
  attribution, while their share of the area is several times larger. The blind spot is plain
  to see — exactly what the lesson describes.
- **Blur IG** gave them more than their share of the area. There is no blind spot at all: on a
  blur path a dark pixel never coincides with the reference point, because the reference point
  is a blurred version of itself rather than the colour black.
- **Guided IG** barely moved. **And that is correct, not a breakdown.** Guided IG changes the
  path, while its baseline stayed zero — so the blind spot stayed too. The method cures a
  different ailment.

**Task 2.** Replace the zero baseline in `guided_ig` with the blurred picture and repeat the
measurement. Does the spot disappear? What does that say about which of the two knobs is
responsible for it?

In [ ]:
# Your code here

## 4. Blur IG has a blind spot too — just in a different place

Before going further it is worth stress-testing the claim "Blur IG fixes the blind spot". The
spot of a black baseline comes from the factor that zeroes the contribution wherever the input
coincides with the reference point. Blur IG has a different reference point — a blurred version
of the picture itself. Let us ask: **can a pixel coincide with that one too?**

It can: in the middle of a large uniform region blurring changes almost nothing. So along the
whole path the value of such a pixel stays put, while the contribution is computed from the
shift along the path — and there will be almost no contribution.

Let us measure it: take the pixels for which the picture and its blurred version are close and
look at their share of the attribution.

In [ ]:
same = (img - blur(img, 20.0)).abs().mean(1, keepdim=True)
for thr in (1e-3, 1e-2):
    mask = (same < thr).float()
    mass = a_blur.abs().sum(1, keepdim=True)
    print(f'   threshold {thr:g}: share of area {mask.mean().item() * 100:5.2f} %   '
          f'share of attribution {(mass * mask).sum().item() / mass.sum().item() * 100:6.3f} %')

The share of attribution for such pixels is **several times smaller than their
share of area** — the same thing we saw for a black baseline on dark regions, only the region
is a different one.

It does not come out exactly zero, and that is explainable too: a blur of finite radius never
leaves a pixel completely unchanged, so the factor is small but not zero. For a black baseline
in a black pixel it vanishes exactly.

**The conclusion worth taking instead of the slogan "Blur IG is better":** a blind spot exists
for any method that defines importance by comparison with a reference. The question is not how
to find a method without a spot, but to know **where** the spot of yours is, and whether that
place coincides with what matters to you in the picture.

## 5. So what does Guided IG fix

Its own ailment: noise. A straight path runs through half-transparent pictures the network has
never seen, gradients jump there, and that jumping settles into the map as speckle. At each step
Guided IG moves only those features whose gradient is small in magnitude — that is, it travels
where the function changes more calmly.

Let us measure the noise as the mean step between neighbouring pixels of the map.

In [ ]:
for name, a in (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided)):
    m = a.abs().sum(1, keepdim=True)
    m = m / m.max()
    tv = ((m[:, :, 1:] - m[:, :, :-1]).abs().mean() + (m[..., 1:] - m[..., :-1]).abs().mean())
    print(f'   {name:11} {tv.item():.5f}')

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (name, a) in zip(axes, (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided))):
    m = a.abs().sum(1)[0]
    ax.imshow((m / m.quantile(0.99)).clamp(0, 1).numpy(), cmap='inferno')
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

Guided IG is noticeably quieter than plain IG — roughly by half. Blur IG is quieter
too, but for a different reason: its path runs through blurred images, and the low-frequency
nature of the path carries over to the map.

**The summary worth taking whole.** Two knobs — two different cures:

| what we change | method | what it fixes | what it does NOT fix |
| --- | --- | --- | --- |
| the reference point | Expected Gradients | the blind spot | path noise |
| the path | Guided IG | noise | the blind spot |
| both | Blur IG | the dark-region spot and noise | a spot of its own on uniform regions |

**Task 3.** Build an Expected Gradients map (the code is in the "Expected Gradients" notebook of
this module) and add it to both tables, the blind spot one and the noise one. Does it land where
its row promises?

In [ ]:
# Your code here

## What to take away from this notebook

- **A path integral has two independent knobs**, and confusing them is costly: Guided IG does
  not replace Expected Gradients, and Blur IG is not "just a better IG".
- **The blind spot is measurable in percentages**, not only describable in words. If your
  objects are sometimes dark, that share is worth computing before you trust the map.
- **Completeness holds for all three**, but the deviation is measured from different reference
  points. A number in a report without a stated reference point means nothing.
- **On the Blur IG path every intermediate point looks like a real image.** That argument is not
  aesthetic: gradients outside the data distribution are noisy, and all of that noise honestly
  ends up in the integral.